# Análise Exploratória de Datasets - Dataset Picking

Este notebook compara três datasets industriais para seleção do melhor para o projeto:
1. **Hydraulic Systems**: Monitoramento de condição de sistemas hidráulicos
2. **Hydropower Turbines**: Monitoramento de fricção em turbinas hidrelétricas  
3. **Industrial Equipment**: Dataset de monitoramento de equipamentos industriais

**Objetivo**: Análise comparativa para escolher o dataset mais adequado para o projeto de IA.

## Importação de Bibliotecas

In [ ]:
# Bibliotecas padrão
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns       
import warnings
import os
from pathlib import Path
from dotenv import load_dotenv

# Configuração para downloads
import kagglehub

project_root = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd().parent
load_dotenv(project_root / '.env')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✓ Bibliotecas importadas com sucesso!")

%matplotlib inline

## Importação de Datasets

### Dataset 1: Hydraulic Systems (Condition Monitoring)
- **Fonte**: Local - `/datasets/Condition monitoring of hydraulic systems/`
- **Formato**: Arquivos TXT (tab-delimited) - um por sensor
- **Instâncias**: 2,205 ciclos de 60 segundos
- **Sensores**: 17 sensores com frequências variadas (1-100 Hz)

In [ ]:
hydraulic_path_env = os.getenv('HYDRAULIC_DATASET_PATH')
hydraulic_path = Path(hydraulic_path_env)
if not hydraulic_path.is_absolute():
    hydraulic_path = (Path.cwd().parent / hydraulic_path_env).resolve()

print(f"📂 Carregando Hydraulic Systems de: {hydraulic_path.absolute()}")

# Primeiro, carregar os targets (profile.txt)
profile_cols = ['cooler_condition', 'valve_condition', 'pump_leakage', 'accumulator', 'stable_flag']
dataset_hydraulic_targets = pd.read_csv(
    hydraulic_path / "profile.txt",
    sep='\t',
    names=profile_cols
)

print("Targets carregados:")
print(f"  Shape: {dataset_hydraulic_targets.shape}")
print(f"  Colunas: {list(dataset_hydraulic_targets.columns)}")
print(f"\nDistribuição dos targets:")
for col in profile_cols:
    print(f"  {col}: {dataset_hydraulic_targets[col].unique()}")

sensor_files_low_freq = {
    'CE': 'CE.txt',     # Cooling efficiency
    'CP': 'CP.txt',     # Cooling power
    'SE': 'SE.txt',     # Efficiency factor
    'TS1': 'TS1.txt',   # Temperature sensor 1
    'TS2': 'TS2.txt',   # Temperature sensor 2
    'TS3': 'TS3.txt',   # Temperature sensor 3
    'TS4': 'TS4.txt',   # Temperature sensor 4
    'VS1': 'VS1.txt'    # Vibration sensor 1
}

print("\n\nCarregando sensores de baixa frequência (1 Hz - 60 amostras/ciclo)...")
sensor_data = {}

for sensor_name, filename in sensor_files_low_freq.items():
    filepath = hydraulic_path / filename
    # Carregar com numpy primeiro
    data = np.loadtxt(filepath, delimiter='\t')
    # Converter para DataFrame com colunas nomeadas
    col_names = [f"{sensor_name}_t{i:03d}" for i in range(data.shape[1])]
    sensor_df = pd.DataFrame(data, columns=col_names)
    sensor_data[sensor_name] = sensor_df
    print(f"  ✓ {sensor_name}: shape {data.shape}")

# Combinar todos os sensores em um único DataFrame
dataset_hydraulic = pd.concat([sensor_data[s] for s in sensor_files_low_freq.keys()], axis=1)

# Adicionar os targets
dataset_hydraulic = pd.concat([dataset_hydraulic, dataset_hydraulic_targets], axis=1)

print(f"\n✓ Dataset Hydraulic Systems carregado!")

print(f"  Shape final: {dataset_hydraulic.shape}")

print(f"  Targets: {len(profile_cols)} variáveis")
print(f"  Features: {dataset_hydraulic.shape[1] - len(profile_cols)} sensores")

### Dataset 2: Hydropower Turbines (Friction Monitoring)
- **Fonte**: Local - `/datasets/Hydropower Turbine Friction Monitoring Dataset/`
- **Formato**: CSV com headers
- **Instâncias**: Turbine_1 (~814k rows), Turbine_2 (~1M rows)
- **Features**: Pressão e temperatura (avg, min, max) + target binário

In [ ]:
hydropower_path_env = os.getenv('HYDROPOWER_DATASET_PATH', '../datasets/Hydropower Turbine Friction Monitoring Dataset/data')
hydropower_path = Path(hydropower_path_env)
if not hydropower_path.is_absolute():
    hydropower_path = (Path.cwd().parent / hydropower_path_env).resolve()

print(f"📂 Carregando Hydropower Turbines de: {hydropower_path.absolute()}")

# Carregar Turbine 1
print("\nCarregando Turbine_1.csv...")
dataset_hydropower_t1 = pd.read_csv(hydropower_path / "Turbine_1.csv")
print(f"  ✓ Turbine 1: {dataset_hydropower_t1.shape}")
print(f"    Colunas: {list(dataset_hydropower_t1.columns)}")

# Carregar Turbine 2
print("\nCarregando Turbine_2.csv...")
dataset_hydropower_t2 = pd.read_csv(hydropower_path / "Turbine_2.csv")
print(f"  ✓ Turbine 2: {dataset_hydropower_t2.shape}")
print(f"    Colunas: {list(dataset_hydropower_t2.columns)}")

print(f"\n✓ Datasets Hydropower carregados!")
print(f"  Total de instâncias: {len(dataset_hydropower_t1) + len(dataset_hydropower_t2):,}")

print(f"  Features Turbine 1: {dataset_hydropower_t1.shape[1]}")  
print(f"  Features Turbine 2: {dataset_hydropower_t2.shape[1]}")  

### Dataset 3: Industrial Equipment Monitoring (Kaggle)
- **Fonte**: Kaggle via kagglehub
- **ID**: `dnkumars/industrial-equipment-monitoring-dataset`
- **Formato**: A ser descoberto após download

In [ ]:
# Download do dataset via kagglehub
print("Baixando dataset do Kaggle...")
path = kagglehub.dataset_download("dnkumars/industrial-equipment-monitoring-dataset")

print(f"✓ Path to dataset files: {path}")

# Explorar o conteúdo do diretório
industrial_files = list(Path(path).rglob("*"))
print(f"\nArquivos encontrados: {len(industrial_files)}")
for f in industrial_files[:10]:  # Mostrar primeiros 10
    if f.is_file():
        size_mb = f.stat().st_size / (1024*1024)
        print(f"  - {f.name} ({size_mb:.2f} MB)")

# Identificar arquivos CSV ou principais
csv_files = list(Path(path).glob("*.csv"))
if not csv_files:
    csv_files = list(Path(path).rglob("*.csv"))

print(f"\nArquivos CSV encontrados: {len(csv_files)}")

# Carregar o principal arquivo (assumindo que existe um CSV principal)
if csv_files:
    main_file = csv_files[0]
    print(f"\nCarregando arquivo principal: {main_file.name}")
    dataset_industrial = pd.read_csv(main_file)
    print(f"  ✓ Shape: {dataset_industrial.shape}")
    print(f"  Colunas: {list(dataset_industrial.columns)}")
else:
    print("⚠ Nenhum arquivo CSV encontrado. Verificar estrutura do dataset.")

---
## Visualização Inicial dos Datasets

Nesta seção vamos explorar a estrutura básica de cada dataset, verificar dimensões, tipos de dados e primeiras linhas.

### Dataset 1: Hydraulic Systems - Visão Geral

In [ ]:
print("="*80)
print("HYDRAULIC SYSTEMS DATASET - ESTRUTURA")
print("="*80)

print(f"\n📊 Shape: {dataset_hydraulic.shape}")
print(f"   - Instâncias (ciclos): {dataset_hydraulic.shape[0]:,}")
print(f"   - Features totais: {dataset_hydraulic.shape[1]:,}")
print(f"   - Sensores de série temporal: {dataset_hydraulic.shape[1] - len(profile_cols):,}")
print(f"   - Variáveis target: {len(profile_cols)}")

print(f"\n📋 Informações do DataFrame:")
print(f"   - Memória utilizada: {dataset_hydraulic.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   - Tipos de dados: {dataset_hydraulic.dtypes.value_counts().to_dict()}")

print(f"\n🔍 Primeiras 5 linhas (apenas targets + algumas features):")
# Mostrar apenas algumas colunas para visualização
cols_to_show = list(dataset_hydraulic.columns[:10]) + profile_cols
display(dataset_hydraulic[cols_to_show].head())

print(f"\n🔍 Últimas 5 linhas (targets + algumas features):")
display(dataset_hydraulic[cols_to_show].tail())

print(f"\n📈 Estatísticas descritivas das variáveis target:")
display(dataset_hydraulic[profile_cols].describe())

### Dataset 2: Hydropower Turbines - Visão Geral

In [ ]:
print("="*80)
print("HYDROPOWER TURBINES DATASET - ESTRUTURA")
print("="*80)

print(f"\n🔧 TURBINE 1:")
print(f"   📊 Shape: {dataset_hydropower_t1.shape}")
print(f"   - Instâncias: {dataset_hydropower_t1.shape[0]:,}")
print(f"   - Features: {dataset_hydropower_t1.shape[1] - 1}")  # -1 para turbine_id
print(f"   - Memória: {dataset_hydropower_t1.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n   🔍 Primeiras 5 linhas:")
display(dataset_hydropower_t1.head())

print(f"\n   🔍 Últimas 5 linhas:")
display(dataset_hydropower_t1.tail())

print(f"\n   📈 Estatísticas descritivas:")
display(dataset_hydropower_t1.describe())

print("\n" + "="*80)

print(f"\n🔧 TURBINE 2:")
print(f"   📊 Shape: {dataset_hydropower_t2.shape}")
print(f"   - Instâncias: {dataset_hydropower_t2.shape[0]:,}")
print(f"   - Features: {dataset_hydropower_t2.shape[1] - 1}")
print(f"   - Memória: {dataset_hydropower_t2.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n   🔍 Primeiras 5 linhas:")
display(dataset_hydropower_t2.head())

print(f"\n   🔍 Últimas 5 linhas:")
display(dataset_hydropower_t2.tail())

print(f"\n   📈 Estatísticas descritivas:")
display(dataset_hydropower_t2.describe())

print(f"\n💡 Observação: Turbine 1 e Turbine 2 têm features diferentes!")
print(f"   - Turbine 1 tem temperatura do óleo (t_oil)")
print(f"   - Turbine 2 tem pressão do sistema (p_sys)")

### Dataset 3: Industrial Equipment - Visão Geral

In [ ]:
print("="*80)
print("INDUSTRIAL EQUIPMENT DATASET - ESTRUTURA")
print("="*80)

if 'dataset_industrial' in locals():
    print(f"\n📊 Shape: {dataset_industrial.shape}")
    print(f"   - Instâncias: {dataset_industrial.shape[0]:,}")
    print(f"   - Features: {dataset_industrial.shape[1]}")
    print(f"   - Memória: {dataset_industrial.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print(f"\n📋 Informações do DataFrame:")
    print(f"   - Colunas: {list(dataset_industrial.columns)}")
    print(f"   - Tipos de dados: {dataset_industrial.dtypes.value_counts().to_dict()}")
    
    print(f"\n🔍 Primeiras 10 linhas:")
    display(dataset_industrial.head(10))
    
    print(f"\n🔍 Últimas 10 linhas:")
    display(dataset_industrial.tail(10))
    
    print(f"\n📈 Estatísticas descritivas:")
    display(dataset_industrial.describe())
else:
    print("⚠ Dataset industrial não carregado. Verifique a célula de download anterior.")

---
## Análise de Valores Ausentes (Missing Values)

Verificando a presença de valores NaN/null em cada dataset.

In [ ]:
print("="*80)
print("ANÁLISE DE VALORES AUSENTES")
print("="*80)

# Dataset 1: Hydraulic Systems
print("\n🔧 DATASET 1: HYDRAULIC SYSTEMS")
missing_hydraulic = dataset_hydraulic.isna().sum()
total_missing_hydraulic = missing_hydraulic.sum()
pct_missing_hydraulic = (total_missing_hydraulic / (dataset_hydraulic.shape[0] * dataset_hydraulic.shape[1])) * 100

print(f"   Total de valores ausentes: {total_missing_hydraulic:,}")
print(f"   Percentual do dataset: {pct_missing_hydraulic:.4f}%")

if total_missing_hydraulic > 0:
    print(f"\n   Colunas com valores ausentes:")
    missing_cols = missing_hydraulic[missing_hydraulic > 0].sort_values(ascending=False)
    for col, count in missing_cols.head(10).items():
        pct = (count / dataset_hydraulic.shape[0]) * 100
        print(f"      - {col}: {count} ({pct:.2f}%)")
else:
    print(f"   ✓ Nenhum valor ausente encontrado!")

# Dataset 2: Hydropower Turbine 1
print("\n🔧 DATASET 2a: HYDROPOWER TURBINE 1")
missing_hydro_t1 = dataset_hydropower_t1.isna().sum()
total_missing_hydro_t1 = missing_hydro_t1.sum()
pct_missing_hydro_t1 = (total_missing_hydro_t1 / (dataset_hydropower_t1.shape[0] * dataset_hydropower_t1.shape[1])) * 100

print(f"   Total de valores ausentes: {total_missing_hydro_t1:,}")
print(f"   Percentual do dataset: {pct_missing_hydro_t1:.4f}%")

if total_missing_hydro_t1 > 0:
    print(f"\n   Colunas com valores ausentes:")
    missing_cols = missing_hydro_t1[missing_hydro_t1 > 0].sort_values(ascending=False)
    for col, count in missing_cols.items():
        pct = (count / dataset_hydropower_t1.shape[0]) * 100
        print(f"      - {col}: {count:,} ({pct:.2f}%)")
else:
    print(f"   ✓ Nenhum valor ausente encontrado!")

# Dataset 2: Hydropower Turbine 2
print("\n🔧 DATASET 2b: HYDROPOWER TURBINE 2")
missing_hydro_t2 = dataset_hydropower_t2.isna().sum()
total_missing_hydro_t2 = missing_hydro_t2.sum()
pct_missing_hydro_t2 = (total_missing_hydro_t2 / (dataset_hydropower_t2.shape[0] * dataset_hydropower_t2.shape[1])) * 100

print(f"   Total de valores ausentes: {total_missing_hydro_t2:,}")
print(f"   Percentual do dataset: {pct_missing_hydro_t2:.4f}%")

if total_missing_hydro_t2 > 0:
    print(f"\n   Colunas com valores ausentes:")
    missing_cols = missing_hydro_t2[missing_hydro_t2 > 0].sort_values(ascending=False)
    for col, count in missing_cols.items():
        pct = (count / dataset_hydropower_t2.shape[0]) * 100
        print(f"      - {col}: {count:,} ({pct:.2f}%)")
else:
    print(f"   ✓ Nenhum valor ausente encontrado!")

# Dataset 3: Industrial Equipment
print("\n🔧 DATASET 3: INDUSTRIAL EQUIPMENT")
if 'dataset_industrial' in locals():
    missing_industrial = dataset_industrial.isna().sum()
    total_missing_industrial = missing_industrial.sum()
    pct_missing_industrial = (total_missing_industrial / (dataset_industrial.shape[0] * dataset_industrial.shape[1])) * 100
    
    print(f"   Total de valores ausentes: {total_missing_industrial:,}")
    print(f"   Percentual do dataset: {pct_missing_industrial:.4f}%")
    
    if total_missing_industrial > 0:
        print(f"\n   Colunas com valores ausentes:")
        missing_cols = missing_industrial[missing_industrial > 0].sort_values(ascending=False)
        for col, count in missing_cols.items():
            pct = (count / dataset_industrial.shape[0]) * 100
            print(f"      - {col}: {count:,} ({pct:.2f}%)")
    else:
        print(f"   ✓ Nenhum valor ausente encontrado!")
else:
    print("   ⚠ Dataset não carregado")

#### Visualização de Valores Ausentes

In [ ]:
# Visualizar missing values para Hydropower (datasets menores em termos de features)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Turbine 1
if dataset_hydropower_t1.isna().sum().sum() > 0:
    sns.heatmap(dataset_hydropower_t1.isna(), cbar=False, yticklabels=False, ax=axes[0], cmap='viridis')
    axes[0].set_title('Hydropower Turbine 1 - Missing Values', fontsize=14, fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'Nenhum valor ausente', ha='center', va='center', fontsize=16)
    axes[0].set_title('Hydropower Turbine 1', fontsize=14, fontweight='bold')
    axes[0].axis('off')

# Turbine 2
if dataset_hydropower_t2.isna().sum().sum() > 0:
    sns.heatmap(dataset_hydropower_t2.isna(), cbar=False, yticklabels=False, ax=axes[1], cmap='viridis')
    axes[1].set_title('Hydropower Turbine 2 - Missing Values', fontsize=14, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'Nenhum valor ausente', ha='center', va='center', fontsize=16)
    axes[1].set_title('Hydropower Turbine 2', fontsize=14, fontweight='bold')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

# Para Hydraulic, seria muito grande para visualizar (480 colunas), então mostramos apenas summary
print("\n💡 Para o dataset Hydraulic Systems, com 480+ features, a visualização de heatmap seria ilegível.")
print("   Os números acima mostram que não há valores ausentes neste dataset.")

---
## Análise de Distribuição de Features

Explorando as distribuições das features principais de cada dataset.

### Dataset 1: Hydraulic Systems - Séries Temporais de Sensores

In [ ]:
# Visualizar séries temporais de diferentes sensores
# Vamos plotar alguns ciclos de exemplo para cada tipo de sensor

fig, axes = plt.subplots(4, 2, figsize=(18, 14))
axes = axes.flatten()

sensors_to_plot = ['CE', 'CP', 'SE', 'TS1', 'TS2', 'TS3', 'TS4', 'VS1']
sample_cycles = [0, 1, 2]  # Plotar primeiros 3 ciclos

for idx, sensor in enumerate(sensors_to_plot):
    # Obter colunas do sensor
    sensor_cols = [col for col in dataset_hydraulic.columns if col.startswith(f"{sensor}_t")]
    
    # Plotar alguns ciclos
    for cycle_idx in sample_cycles:
        values = dataset_hydraulic.loc[cycle_idx, sensor_cols].values
        time_points = np.arange(len(values))
        axes[idx].plot(time_points, values, label=f'Ciclo {cycle_idx}', alpha=0.7)
    
    axes[idx].set_title(f'{sensor} - Séries Temporais (3 ciclos)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Tempo (amostras)')
    axes[idx].set_ylabel('Valor')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observações:")
print("   - Cada sensor captura um ciclo de 60 segundos")
print("   - CE: Cooling efficiency")
print("   - CP: Cooling power")
print("   - SE: Efficiency factor")
print("   - TS1-TS4: Sensores de temperatura")
print("   - VS1: Sensor de vibração")

### Dataset 2: Hydropower Turbines - Distribuições de Features

In [ ]:
# Turbine 1 - Histogramas e box plots
numeric_cols_t1 = dataset_hydropower_t1.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_t1 = [col for col in numeric_cols_t1 if col != 'turbine_id']

fig, axes = plt.subplots(len(numeric_cols_t1), 2, figsize=(16, len(numeric_cols_t1) * 3))

for idx, col in enumerate(numeric_cols_t1):
    # Histograma
    axes[idx, 0].hist(dataset_hydropower_t1[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='skyblue')
    axes[idx, 0].set_title(f'Turbine 1 - {col} (Histograma)', fontsize=11, fontweight='bold')
    axes[idx, 0].set_xlabel('Valor')
    axes[idx, 0].set_ylabel('Frequência')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Box plot
    axes[idx, 1].boxplot(dataset_hydropower_t1[col].dropna(), vert=False)
    axes[idx, 1].set_title(f'Turbine 1 - {col} (Box Plot)', fontsize=11, fontweight='bold')
    axes[idx, 1].set_xlabel('Valor')
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)

# Turbine 2 - Histogramas e box plots
numeric_cols_t2 = dataset_hydropower_t2.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_t2 = [col for col in numeric_cols_t2 if col != 'turbine_id']

fig, axes = plt.subplots(len(numeric_cols_t2), 2, figsize=(16, len(numeric_cols_t2) * 3))

for idx, col in enumerate(numeric_cols_t2):
    # Histograma
    axes[idx, 0].hist(dataset_hydropower_t2[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='lightcoral')
    axes[idx, 0].set_title(f'Turbine 2 - {col} (Histograma)', fontsize=11, fontweight='bold')
    axes[idx, 0].set_xlabel('Valor')
    axes[idx, 0].set_ylabel('Frequência')
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Box plot
    axes[idx, 1].boxplot(dataset_hydropower_t2[col].dropna(), vert=False)
    axes[idx, 1].set_title(f'Turbine 2 - {col} (Box Plot)', fontsize=11, fontweight='bold')
    axes[idx, 1].set_xlabel('Valor')
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Dataset 3: Industrial Equipment - Distribuições de Features

In [ ]:
# Industrial Equipment - Histogramas e box plots
if 'dataset_industrial' in locals():
    numeric_cols_industrial = dataset_industrial.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(numeric_cols_industrial) > 0:
        print(f"📊 Visualizando {len(numeric_cols_industrial)} features numéricas do dataset Industrial Equipment")
        
        fig, axes = plt.subplots(len(numeric_cols_industrial), 2, figsize=(16, len(numeric_cols_industrial) * 3))
        
        # Se houver apenas uma feature, axes não será 2D
        if len(numeric_cols_industrial) == 1:
            axes = axes.reshape(1, -1)
        
        for idx, col in enumerate(numeric_cols_industrial):
            # Histograma
            axes[idx, 0].hist(dataset_industrial[col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='mediumpurple')
            axes[idx, 0].set_title(f'Industrial Equipment - {col} (Histograma)', fontsize=11, fontweight='bold')
            axes[idx, 0].set_xlabel('Valor')
            axes[idx, 0].set_ylabel('Frequência')
            axes[idx, 0].grid(True, alpha=0.3)
            
            # Box plot
            axes[idx, 1].boxplot(dataset_industrial[col].dropna(), vert=False)
            axes[idx, 1].set_title(f'Industrial Equipment - {col} (Box Plot)', fontsize=11, fontweight='bold')
            axes[idx, 1].set_xlabel('Valor')
            axes[idx, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("\n💡 Estatísticas descritivas:")
        display(dataset_industrial[numeric_cols_industrial].describe())
    else:
        print("⚠️ Nenhuma coluna numérica encontrada no dataset Industrial Equipment")
else:
    print("⚠️ Dataset Industrial Equipment não carregado. Execute a célula de download primeiro.")

---
## Análise de Correlação

Explorando as correlações entre features e com as variáveis target.

### Dataset 1: Hydraulic Systems - Correlação entre Agregados de Sensores

In [ ]:
# Para o dataset hydraulic com 480+ features de séries temporais, vamos agregar
# Calcular estatísticas por sensor: média, std, min, max

sensors = ['CE', 'CP', 'SE', 'TS1', 'TS2', 'TS3', 'TS4', 'VS1']
aggregated_features = {}

for sensor in sensors:
    sensor_cols = [col for col in dataset_hydraulic.columns if col.startswith(f"{sensor}_t")]
    sensor_data = dataset_hydraulic[sensor_cols]
    
    aggregated_features[f'{sensor}_mean'] = sensor_data.mean(axis=1)
    aggregated_features[f'{sensor}_std'] = sensor_data.std(axis=1)
    aggregated_features[f'{sensor}_min'] = sensor_data.min(axis=1)
    aggregated_features[f'{sensor}_max'] = sensor_data.max(axis=1)

# Criar DataFrame com features agregadas + targets
df_aggregated = pd.DataFrame(aggregated_features)
df_aggregated = pd.concat([df_aggregated, dataset_hydraulic[profile_cols]], axis=1)

print(f"✓ Features agregadas criadas: {df_aggregated.shape}")
print(f"  - {len(sensors)} sensores × 4 estatísticas = {len(sensors)*4} features + {len(profile_cols)} targets")

# Calcular correlação
corr_matrix = df_aggregated.corr()

# Visualizar matriz de correlação completa
plt.figure(figsize=(18, 16))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Hydraulic Systems - Matriz de Correlação (Features Agregadas + Targets)', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Mostrar correlações mais fortes com os targets
print("\n" + "="*80)
print("CORRELAÇÕES MAIS FORTES COM OS TARGETS")
print("="*80)

for target in profile_cols:
    print(f"\n🎯 Target: {target}")
    correlations = corr_matrix[target].drop(profile_cols).abs().sort_values(ascending=False)
    print(f"   Top 5 features correlacionadas:")
    for feature, corr_value in correlations.head(5).items():
        actual_corr = corr_matrix[target][feature]
        print(f"      - {feature}: {actual_corr:.4f}")

### Dataset 2: Hydropower Turbines - Matriz de Correlação

In [ ]:
# Turbine 1 - Correlação
numeric_t1 = dataset_hydropower_t1.select_dtypes(include=[np.number])
corr_t1 = numeric_t1.corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Turbine 1
sns.heatmap(corr_t1, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, ax=axes[0], fmt='.2f')
axes[0].set_title('Turbine 1 - Matriz de Correlação', fontsize=14, fontweight='bold')

# Turbine 2 - Correlação
numeric_t2 = dataset_hydropower_t2.select_dtypes(include=[np.number])
corr_t2 = numeric_t2.corr()

sns.heatmap(corr_t2, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, ax=axes[1], fmt='.2f')
axes[1].set_title('Turbine 2 - Matriz de Correlação', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Análise de correlação com target
print("="*80)
print("CORRELAÇÕES COM O TARGET (Glycerol)")
print("="*80)

print("\n🔧 TURBINE 1:")
if 'Glycerol' in dataset_hydropower_t1.columns:
    correlations_t1 = corr_t1['Glycerol'].drop('Glycerol').abs().sort_values(ascending=False)
    print("   Features mais correlacionadas com Glycerol:")
    for feature, corr_value in correlations_t1.items():
        actual_corr = corr_t1['Glycerol'][feature]
        print(f"      - {feature}: {actual_corr:.4f}")

print("\n🔧 TURBINE 2:")
if 'Glycerol' in dataset_hydropower_t2.columns:
    correlations_t2 = corr_t2['Glycerol'].drop('Glycerol').abs().sort_values(ascending=False)
    print("   Features mais correlacionadas com Glycerol:")
    for feature, corr_value in correlations_t2.items():
        actual_corr = corr_t2['Glycerol'][feature]
        print(f"      - {feature}: {actual_corr:.4f}")

### Dataset 3: Industrial Equipment - Matriz de Correlação

In [ ]:
# Industrial Equipment - Análise de correlação
if 'dataset_industrial' in locals():
    numeric_industrial = dataset_industrial.select_dtypes(include=[np.number])
    
    if len(numeric_industrial.columns) > 1:
        corr_industrial = numeric_industrial.corr()
        
        # Visualizar matriz de correlação
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_industrial, annot=True, cmap='coolwarm', center=0, 
                    square=True, linewidths=0.5, fmt='.2f', cbar_kws={"shrink": 0.8})
        plt.title('Industrial Equipment - Matriz de Correlação', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Análise de correlação com target (se existir)
        print("="*80)
        print("ANÁLISE DE CORRELAÇÃO - INDUSTRIAL EQUIPMENT")
        print("="*80)
        
        # Tentar identificar coluna target (pode ser 'target', 'label', 'class', etc.)
        possible_target_names = ['target', 'label', 'class', 'failure', 'status', 'condition', 
                                 'Target', 'Label', 'Class', 'Failure', 'Status', 'Condition']
        target_col = None
        
        for col_name in possible_target_names:
            if col_name in dataset_industrial.columns:
                target_col = col_name
                break
        
        if target_col:
            print(f"\n🎯 Target identificado: '{target_col}'")
            if target_col in corr_industrial.columns:
                correlations_industrial = corr_industrial[target_col].drop(target_col).abs().sort_values(ascending=False)
                print(f"   Top 10 features correlacionadas com {target_col}:")
                for feature, corr_value in correlations_industrial.head(10).items():
                    actual_corr = corr_industrial[target_col][feature]
                    print(f"      - {feature}: {actual_corr:.4f}")
        else:
            print("\n💡 Nenhuma coluna target óbvia identificada.")
            print("   Mostrando pares de features com maior correlação:")
            
            # Obter matriz triangular superior
            corr_tri = corr_industrial.where(np.triu(np.ones(corr_industrial.shape), k=1).astype(bool))
            correlations_pairs = corr_tri.unstack().dropna().abs().sort_values(ascending=False)
            
            print("\n   Top 10 pares de features mais correlacionados:")
            for (col1, col2), corr_value in correlations_pairs.head(10).items():
                actual_corr = corr_industrial.loc[col1, col2]
                print(f"      - {col1} ↔ {col2}: {actual_corr:.4f}")
    else:
        print("⚠️ Dataset possui apenas uma coluna numérica, correlação não aplicável")
else:
    print("⚠️ Dataset Industrial Equipment não carregado. Execute a célula de download primeiro.")

---
## Análise das Variáveis Target

Explorando as distribuições e balanceamento das variáveis alvo de cada dataset.

### Dataset 1: Hydraulic Systems - Análise dos Targets

In [ ]:
# Visualizar distribuição dos targets
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, target in enumerate(profile_cols):
    value_counts = dataset_hydraulic[target].value_counts().sort_index()
    
    # Bar plot
    axes[idx].bar(value_counts.index.astype(str), value_counts.values, 
                  color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{target}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Classe')
    axes[idx].set_ylabel('Frequência')
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Adicionar valores no topo das barras
    for i, (cls, count) in enumerate(value_counts.items()):
        pct = (count / len(dataset_hydraulic)) * 100
        axes[idx].text(i, count, f'{count}\n({pct:.1f}%)', 
                      ha='center', va='bottom', fontsize=9)

# Remover eixo extra se houver
if len(profile_cols) < 6:
    fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

# Estatísticas de balanceamento
print("="*80)
print("BALANCEAMENTO DAS CLASSES - HYDRAULIC SYSTEMS")
print("="*80)

for target in profile_cols:
    print(f"\n🎯 {target}:")
    value_counts = dataset_hydraulic[target].value_counts().sort_index()
    print(f"   Classes: {list(value_counts.index)}")
    print(f"   Distribuição:")
    for cls, count in value_counts.items():
        pct = (count / len(dataset_hydraulic)) * 100
        print(f"      - Classe {cls}: {count} ({pct:.2f}%)")
    
    # Calcular desbalanceamento (razão entre maior e menor classe)
    max_count = value_counts.max()
    min_count = value_counts.min()
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
    print(f"   Razão de desbalanceamento: {imbalance_ratio:.2f}:1")

### Dataset 2: Hydropower Turbines - Análise do Target (Glycerol)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Turbine 1
if 'Glycerol' in dataset_hydropower_t1.columns:
    value_counts_t1 = dataset_hydropower_t1['Glycerol'].value_counts().sort_index()
    axes[0].bar(value_counts_t1.index.astype(str), value_counts_t1.values,
                color=['#2ecc71', '#e74c3c'], edgecolor='black', alpha=0.7)
    axes[0].set_title('Turbine 1 - Distribuição do Target (Glycerol)', 
                      fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Classe (0=Sem Glycerol, 1=Com Glycerol)')
    axes[0].set_ylabel('Frequência')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    for i, (cls, count) in enumerate(value_counts_t1.items()):
        pct = (count / len(dataset_hydropower_t1)) * 100
        axes[0].text(i, count, f'{count:,}\n({pct:.1f}%)', 
                    ha='center', va='bottom', fontsize=10)

# Turbine 2
if 'Glycerol' in dataset_hydropower_t2.columns:
    value_counts_t2 = dataset_hydropower_t2['Glycerol'].value_counts().sort_index()
    axes[1].bar(value_counts_t2.index.astype(str), value_counts_t2.values,
                color=['#2ecc71', '#e74c3c'], edgecolor='black', alpha=0.7)
    axes[1].set_title('Turbine 2 - Distribuição do Target (Glycerol)', 
                      fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Classe (0=Sem Glycerol, 1=Com Glycerol)')
    axes[1].set_ylabel('Frequência')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    for i, (cls, count) in enumerate(value_counts_t2.items()):
        pct = (count / len(dataset_hydropower_t2)) * 100
        axes[1].text(i, count, f'{count:,}\n({pct:.1f}%)', 
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Estatísticas
print("="*80)
print("BALANCEAMENTO DAS CLASSES - HYDROPOWER TURBINES")
print("="*80)

print("\n🔧 TURBINE 1:")
if 'Glycerol' in dataset_hydropower_t1.columns:
    value_counts_t1 = dataset_hydropower_t1['Glycerol'].value_counts().sort_index()
    print(f"   Total de instâncias: {len(dataset_hydropower_t1):,}")
    for cls, count in value_counts_t1.items():
        pct = (count / len(dataset_hydropower_t1)) * 100
        label = "Sem Glycerol" if cls == 0 else "Com Glycerol"
        print(f"      - Classe {cls} ({label}): {count:,} ({pct:.2f}%)")
    
    if len(value_counts_t1) == 2:
        imbalance_ratio = value_counts_t1.max() / value_counts_t1.min()
        print(f"   Razão de desbalanceamento: {imbalance_ratio:.2f}:1")

print("\n🔧 TURBINE 2:")
if 'Glycerol' in dataset_hydropower_t2.columns:
    value_counts_t2 = dataset_hydropower_t2['Glycerol'].value_counts().sort_index()
    print(f"   Total de instâncias: {len(dataset_hydropower_t2):,}")
    for cls, count in value_counts_t2.items():
        pct = (count / len(dataset_hydropower_t2)) * 100
        label = "Sem Glycerol" if cls == 0 else "Com Glycerol"
        print(f"      - Classe {cls} ({label}): {count:,} ({pct:.2f}%)")
    
    if len(value_counts_t2) == 2:
        imbalance_ratio = value_counts_t2.max() / value_counts_t2.min()
        print(f"   Razão de desbalanceamento: {imbalance_ratio:.2f}:1")

### Dataset 3: Industrial Equipment - Análise do Target

In [ ]:
# Industrial Equipment - Análise de target
if 'dataset_industrial' in locals():
    # Tentar identificar coluna target
    possible_target_names = ['target', 'label', 'class', 'failure', 'status', 'condition',
                             'Target', 'Label', 'Class', 'Failure', 'Status', 'Condition']
    target_col = None
    
    for col_name in possible_target_names:
        if col_name in dataset_industrial.columns:
            target_col = col_name
            break
    
    if target_col:
        print(f"🎯 Target identificado: '{target_col}'")
        print("="*80)
        
        # Verificar se é categórico ou numérico
        if dataset_industrial[target_col].dtype == 'object' or dataset_industrial[target_col].nunique() < 20:
            # Target categórico/classificação
            value_counts = dataset_industrial[target_col].value_counts().sort_index()
            
            # Visualização
            plt.figure(figsize=(12, 6))
            colors = plt.cm.viridis(np.linspace(0, 1, len(value_counts)))
            bars = plt.bar(range(len(value_counts)), value_counts.values, 
                          color=colors, edgecolor='black', alpha=0.7)
            plt.xticks(range(len(value_counts)), value_counts.index.astype(str), rotation=45)
            plt.title(f'Industrial Equipment - Distribuição do Target ({target_col})', 
                     fontsize=14, fontweight='bold')
            plt.xlabel('Classe')
            plt.ylabel('Frequência')
            plt.grid(True, alpha=0.3, axis='y')
            
            # Adicionar valores
            for i, (idx, count) in enumerate(value_counts.items()):
                pct = (count / len(dataset_industrial)) * 100
                plt.text(i, count, f'{count:,}\n({pct:.1f}%)', 
                        ha='center', va='bottom', fontsize=9)
            
            plt.tight_layout()
            plt.show()
            
            # Estatísticas
            print("\n" + "="*80)
            print("BALANCEAMENTO DAS CLASSES - INDUSTRIAL EQUIPMENT")
            print("="*80)
            
            print(f"\n📊 Total de instâncias: {len(dataset_industrial):,}")
            print(f"   Número de classes: {len(value_counts)}")
            print(f"\n   Distribuição:")
            for cls, count in value_counts.items():
                pct = (count / len(dataset_industrial)) * 100
                print(f"      - Classe {cls}: {count:,} ({pct:.2f}%)")
            
            # Calcular desbalanceamento
            if len(value_counts) > 1:
                max_count = value_counts.max()
                min_count = value_counts.min()
                imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
                print(f"\n   Razão de desbalanceamento: {imbalance_ratio:.2f}:1")
                
                if imbalance_ratio > 10:
                    print("   ⚠️ ALTA desbalanceamento detectado!")
                elif imbalance_ratio > 3:
                    print("   ⚠️ Desbalanceamento moderado")
                else:
                    print("   ✓ Balanceamento razoável")
        else:
            # Target numérico/regressão
            plt.figure(figsize=(14, 5))
            
            # Histograma
            plt.subplot(1, 2, 1)
            plt.hist(dataset_industrial[target_col].dropna(), bins=50, 
                    edgecolor='black', alpha=0.7, color='mediumpurple')
            plt.title(f'{target_col} - Distribuição', fontsize=12, fontweight='bold')
            plt.xlabel('Valor')
            plt.ylabel('Frequência')
            plt.grid(True, alpha=0.3)
            
            # Box plot
            plt.subplot(1, 2, 2)
            plt.boxplot(dataset_industrial[target_col].dropna(), vert=True)
            plt.title(f'{target_col} - Box Plot', fontsize=12, fontweight='bold')
            plt.ylabel('Valor')
            plt.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            print("\n" + "="*80)
            print("ESTATÍSTICAS DO TARGET - INDUSTRIAL EQUIPMENT")
            print("="*80)
            print(f"\n📊 Estatísticas descritivas de '{target_col}':")
            display(dataset_industrial[target_col].describe())
    else:
        print("⚠️ Nenhuma coluna target óbvia identificada no dataset Industrial Equipment")
        print("   Colunas disponíveis:", list(dataset_industrial.columns))
        print("\n💡 Se houver uma coluna target, adicione-a manualmente na análise")
else:
    print("⚠️ Dataset Industrial Equipment não carregado. Execute a célula de download primeiro.")

---
## 📊 Aplicabilidade para Análise de Séries Temporais

Avaliando qual dataset é mais adequado para técnicas de análise de séries temporais.

In [ ]:
print("="*100)
print("AVALIAÇÃO DE APLICABILIDADE PARA SÉRIES TEMPORAIS")
print("="*100)

# Criar tabela de avaliação
time_series_analysis = {
    'Dataset': [],
    'Tipo de Dado': [],
    'Frequência Temporal': [],
    'Sequências Disponíveis': [],
    'Aplicabilidade': [],
    'Técnicas Recomendadas': []
}

# Dataset 1: Hydraulic Systems
time_series_analysis['Dataset'].append('🔧 Hydraulic Systems')
time_series_analysis['Tipo de Dado'].append('Séries temporais puras')
time_series_analysis['Frequência Temporal'].append('1-100 Hz (60 amostras/ciclo)')
time_series_analysis['Sequências Disponíveis'].append(f'{dataset_hydraulic.shape[0]:,} ciclos × 17 sensores')
time_series_analysis['Aplicabilidade'].append('⭐⭐⭐⭐⭐ EXCELENTE')
time_series_analysis['Técnicas Recomendadas'].append('LSTM, GRU, TCN, Transformers, CNN 1D, Wavelets, FFT')

# Dataset 2: Hydropower Turbines
time_series_analysis['Dataset'].append('⚡ Hydropower T1')
time_series_analysis['Tipo de Dado'].append('Agregados de séries temporais')
time_series_analysis['Frequência Temporal'].append('Agregações (avg, min, max)')
time_series_analysis['Sequências Disponíveis'].append(f'{dataset_hydropower_t1.shape[0]:,} medições')
time_series_analysis['Aplicabilidade'].append('⭐⭐ LIMITADA')
time_series_analysis['Técnicas Recomendadas'].append('Modelos clássicos, Feature Engineering manual')

time_series_analysis['Dataset'].append('⚡ Hydropower T2')
time_series_analysis['Tipo de Dado'].append('Agregados de séries temporais')
time_series_analysis['Frequência Temporal'].append('Agregações (avg, min, max)')
time_series_analysis['Sequências Disponíveis'].append(f'{dataset_hydropower_t2.shape[0]:,} medições')
time_series_analysis['Aplicabilidade'].append('⭐⭐ LIMITADA')
time_series_analysis['Técnicas Recomendadas'].append('Modelos clássicos, Feature Engineering manual')

# Dataset 3: Industrial Equipment
if 'dataset_industrial' in locals():
    # Tentar determinar se há componente temporal
    has_time_column = any(col.lower() in ['time', 'timestamp', 'date', 'datetime', 't'] 
                          for col in dataset_industrial.columns)
    has_sequence_id = any(col.lower() in ['id', 'sequence', 'cycle', 'session'] 
                         for col in dataset_industrial.columns)
    
    if has_time_column or has_sequence_id:
        time_series_analysis['Dataset'].append('🏭 Industrial Equipment')
        time_series_analysis['Tipo de Dado'].append('Possível componente temporal')
        time_series_analysis['Frequência Temporal'].append('A determinar')
        time_series_analysis['Sequências Disponíveis'].append(f'{dataset_industrial.shape[0]:,} registros')
        time_series_analysis['Aplicabilidade'].append('⭐⭐⭐ BOA (a confirmar)')
        time_series_analysis['Técnicas Recomendadas'].append('Depende da estrutura temporal')
    else:
        time_series_analysis['Dataset'].append('🏭 Industrial Equipment')
        time_series_analysis['Tipo de Dado'].append('Dados tabulares agregados')
        time_series_analysis['Frequência Temporal'].append('Sem componente temporal evidente')
        time_series_analysis['Sequências Disponíveis'].append(f'{dataset_industrial.shape[0]:,} registros')
        time_series_analysis['Aplicabilidade'].append('⭐ MÍNIMA')
        time_series_analysis['Técnicas Recomendadas'].append('ML clássico (RF, XGBoost, etc)')
else:
    time_series_analysis['Dataset'].append('🏭 Industrial Equipment')
    time_series_analysis['Tipo de Dado'].append('Não carregado')
    time_series_analysis['Frequência Temporal'].append('N/A')
    time_series_analysis['Sequências Disponíveis'].append('N/A')
    time_series_analysis['Aplicabilidade'].append('N/A')
    time_series_analysis['Técnicas Recomendadas'].append('N/A')

# Criar DataFrame
df_timeseries = pd.DataFrame(time_series_analysis)
df_timeseries = df_timeseries.set_index('Dataset')

print("\n")
display(df_timeseries)

print("\n" + "="*100)
print("RECOMENDAÇÕES PARA SÉRIES TEMPORAIS")
print("="*100)

print("\n🏆 MELHOR ESCOLHA PARA SÉRIES TEMPORAIS: Hydraulic Systems")
print("\n✅ Vantagens:")
print("   • Séries temporais verdadeiras com alta resolução temporal")
print("   • 17 sensores simultâneos (análise multivariada)")
print("   • Dados brutos não processados (máxima informação)")
print("   • Ideal para Deep Learning temporal (LSTM, CNNs, Transformers)")
print("   • Permite análise de frequência (FFT, Wavelets)")
print("   • Detecção de padrões e anomalias temporais")

print("\n⚠️ LIMITAÇÃO PARA SÉRIES TEMPORAIS: Hydropower Turbines")
print("\n❌ Limitações:")
print("   • Dados já agregados (perdeu informação temporal)")
print("   • Apenas estatísticas (avg, min, max) ao invés de sequências")
print("   • Não permite análise de padrões temporais")
print("   • Requer Feature Engineering manual para criar contexto temporal")

print("\n💡 Dicas:")
print("   • Se o objetivo é SÉRIES TEMPORAIS → escolha Hydraulic Systems")
print("   • Se o objetivo é CLASSIFICAÇÃO EM ESCALA → escolha Hydropower")
print("   • Para Hydraulic, considere arquiteturas: CNN-LSTM, Attention, Transformer")
print("   • Para Hydropower, considere: XGBoost, Random Forest, Neural Networks clássicas")

---
---
# 📊 Análise Comparativa Final - Dataset Picking

Resumo comparativo dos três datasets para auxiliar na escolha do melhor para o projeto.

In [ ]:
# Criar tabela comparativa
comparison_data = {
    'Aspecto': [],
    'Hydraulic Systems': [],
    'Hydropower Turbine 1': [],
    'Hydropower Turbine 2': [],
    'Industrial Equipment': []
}

# 1. Tamanho do dataset
comparison_data['Aspecto'].append('📦 Instâncias')
comparison_data['Hydraulic Systems'].append(f"{dataset_hydraulic.shape[0]:,}")
comparison_data['Hydropower Turbine 1'].append(f"{dataset_hydropower_t1.shape[0]:,}")
comparison_data['Hydropower Turbine 2'].append(f"{dataset_hydropower_t2.shape[0]:,}")
comparison_data['Industrial Equipment'].append('N/A' if 'dataset_industrial' not in locals() else f"{dataset_industrial.shape[0]:,}")

# 2. Número de features
comparison_data['Aspecto'].append('🔢 Features (originais)')
comparison_data['Hydraulic Systems'].append(f"{dataset_hydraulic.shape[1] - len(profile_cols):,} (séries temporais)")
comparison_data['Hydropower Turbine 1'].append(f"{dataset_hydropower_t1.shape[1] - 1}")
comparison_data['Hydropower Turbine 2'].append(f"{dataset_hydropower_t2.shape[1] - 1}")
comparison_data['Industrial Equipment'].append('N/A' if 'dataset_industrial' not in locals() else f"{dataset_industrial.shape[1]}")

# 3. Memória
comparison_data['Aspecto'].append('💾 Memória (MB)')
comparison_data['Hydraulic Systems'].append(f"{dataset_hydraulic.memory_usage(deep=True).sum() / 1024**2:.1f}")
comparison_data['Hydropower Turbine 1'].append(f"{dataset_hydropower_t1.memory_usage(deep=True).sum() / 1024**2:.1f}")
comparison_data['Hydropower Turbine 2'].append(f"{dataset_hydropower_t2.memory_usage(deep=True).sum() / 1024**2:.1f}")
comparison_data['Industrial Equipment'].append('N/A' if 'dataset_industrial' not in locals() else f"{dataset_industrial.memory_usage(deep=True).sum() / 1024**2:.1f}")

# 4. Valores ausentes
comparison_data['Aspecto'].append('❌ Missing Values')
comparison_data['Hydraulic Systems'].append(f"{dataset_hydraulic.isna().sum().sum():,}")
comparison_data['Hydropower Turbine 1'].append(f"{dataset_hydropower_t1.isna().sum().sum():,}")
comparison_data['Hydropower Turbine 2'].append(f"{dataset_hydropower_t2.isna().sum().sum():,}")
comparison_data['Industrial Equipment'].append('N/A' if 'dataset_industrial' not in locals() else f"{dataset_industrial.isna().sum().sum():,}")

# 5. Tipo de target
comparison_data['Aspecto'].append('🎯 Tipo de Target')
comparison_data['Hydraulic Systems'].append('Multi-target (4 condições)')
comparison_data['Hydropower Turbine 1'].append('Binário (Glycerol)')
comparison_data['Hydropower Turbine 2'].append('Binário (Glycerol)')
comparison_data['Industrial Equipment'].append('N/A' if 'dataset_industrial' not in locals() else 'A determinar')

# 6. Número de classes
comparison_data['Aspecto'].append('📊 Classes por Target')
comparison_data['Hydraulic Systems'].append('3-4 classes por target')
if 'Glycerol' in dataset_hydropower_t1.columns:
    n_classes_t1 = dataset_hydropower_t1['Glycerol'].nunique()
    comparison_data['Hydropower Turbine 1'].append(f"{n_classes_t1} classes")
else:
    comparison_data['Hydropower Turbine 1'].append('N/A')
if 'Glycerol' in dataset_hydropower_t2.columns:
    n_classes_t2 = dataset_hydropower_t2['Glycerol'].nunique()
    comparison_data['Hydropower Turbine 2'].append(f"{n_classes_t2} classes")
else:
    comparison_data['Hydropower Turbine 2'].append('N/A')
comparison_data['Industrial Equipment'].append('N/A')

# 7. Balanceamento
comparison_data['Aspecto'].append('⚖️ Desbalanceamento')
# Para hydraulic, pegar o pior caso
max_imbalance_hydraulic = 0
for target in profile_cols:
    vc = dataset_hydraulic[target].value_counts()
    imb = vc.max() / vc.min() if vc.min() > 0 else float('inf')
    max_imbalance_hydraulic = max(max_imbalance_hydraulic, imb)
comparison_data['Hydraulic Systems'].append(f"Até {max_imbalance_hydraulic:.1f}:1")

if 'Glycerol' in dataset_hydropower_t1.columns:
    vc_t1 = dataset_hydropower_t1['Glycerol'].value_counts()
    imb_t1 = vc_t1.max() / vc_t1.min() if len(vc_t1) > 1 and vc_t1.min() > 0 else 1.0
    comparison_data['Hydropower Turbine 1'].append(f"{imb_t1:.1f}:1")
else:
    comparison_data['Hydropower Turbine 1'].append('N/A')

if 'Glycerol' in dataset_hydropower_t2.columns:
    vc_t2 = dataset_hydropower_t2['Glycerol'].value_counts()
    imb_t2 = vc_t2.max() / vc_t2.min() if len(vc_t2) > 1 and vc_t2.min() > 0 else 1.0
    comparison_data['Hydropower Turbine 2'].append(f"{imb_t2:.1f}:1")
else:
    comparison_data['Hydropower Turbine 2'].append('N/A')
comparison_data['Industrial Equipment'].append('N/A')

# 8. Complexidade dos dados
comparison_data['Aspecto'].append('🧩 Complexidade')
comparison_data['Hydraulic Systems'].append('Alta (séries temporais)')
comparison_data['Hydropower Turbine 1'].append('Baixa (agregados)')
comparison_data['Hydropower Turbine 2'].append('Baixa (agregados)')
comparison_data['Industrial Equipment'].append('N/A')

# 9. Domínio
comparison_data['Aspecto'].append('🏭 Domínio')
comparison_data['Hydraulic Systems'].append('Sistema hidráulico teste')
comparison_data['Hydropower Turbine 1'].append('Turbina hidrelétrica real')
comparison_data['Hydropower Turbine 2'].append('Turbina hidrelétrica real')
comparison_data['Industrial Equipment'].append('Equipamento industrial')

# Criar DataFrame e exibir
df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.set_index('Aspecto')

print("="*100)
print("TABELA COMPARATIVA DE DATASETS")
print("="*100)
display(df_comparison)

# Visualização gráfica das métricas principais
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Preparar dados com ou sem dataset industrial
if 'dataset_industrial' in locals():
    datasets = ['Hydraulic\nSystems', 'Hydropower\nT1', 'Hydropower\nT2', 'Industrial\nEquipment']
    colors = ['steelblue', 'orange', 'green', 'mediumpurple']
    
    instances = [
        dataset_hydraulic.shape[0],
        dataset_hydropower_t1.shape[0],
        dataset_hydropower_t2.shape[0],
        dataset_industrial.shape[0]
    ]
    
    features = [
        dataset_hydraulic.shape[1] - len(profile_cols),
        dataset_hydropower_t1.shape[1] - 1,
        dataset_hydropower_t2.shape[1] - 1,
        dataset_industrial.shape[1]
    ]
    
    memory = [
        dataset_hydraulic.memory_usage(deep=True).sum() / 1024**2,
        dataset_hydropower_t1.memory_usage(deep=True).sum() / 1024**2,
        dataset_hydropower_t2.memory_usage(deep=True).sum() / 1024**2,
        dataset_industrial.memory_usage(deep=True).sum() / 1024**2
    ]
else:
    datasets = ['Hydraulic\nSystems', 'Hydropower\nT1', 'Hydropower\nT2']
    colors = ['steelblue', 'orange', 'green']
    
    instances = [
        dataset_hydraulic.shape[0],
        dataset_hydropower_t1.shape[0],
        dataset_hydropower_t2.shape[0]
    ]
    
    features = [
        dataset_hydraulic.shape[1] - len(profile_cols),
        dataset_hydropower_t1.shape[1] - 1,
        dataset_hydropower_t2.shape[1] - 1
    ]
    
    memory = [
        dataset_hydraulic.memory_usage(deep=True).sum() / 1024**2,
        dataset_hydropower_t1.memory_usage(deep=True).sum() / 1024**2,
        dataset_hydropower_t2.memory_usage(deep=True).sum() / 1024**2
    ]

# Gráfico 1: Número de instâncias
axes[0].bar(datasets, instances, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_ylabel('Número de Instâncias')
axes[0].set_title('Tamanho dos Datasets', fontsize=13, fontweight='bold')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(instances):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9, rotation=0)

# Gráfico 2: Número de features
axes[1].bar(datasets, features, color=colors, edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Número de Features')
axes[1].set_title('Dimensionalidade', fontsize=13, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(features):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

# Gráfico 3: Memória
axes[2].bar(datasets, memory, color=colors, edgecolor='black', alpha=0.7)
axes[2].set_ylabel('Memória (MB)')
axes[2].set_title('Uso de Memória', fontsize=13, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(memory):
    axes[2].text(i, v, f'{v:.1f} MB', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 🎯 Recomendações para Escolha do Dataset

### ✅ **Dataset 1: Hydraulic Systems**
**Vantagens:**
- 🔬 **Dados ricos**: Séries temporais de alta frequência (1-100 Hz) com 480+ features
- 🎯 **Multi-target**: 4 variáveis de condição diferentes para prever
- 🧪 **Ambiente controlado**: Dados de laboratório, bem documentados
- 📊 **Múltiplas tarefas**: Permite classificação multi-classe e multi-target
- 🤖 **Desafiador para IA**: Requer feature engineering, análise temporal, deep learning

**Desvantagens:**
- ⚙️ **Alta complexidade**: Requer processamento de séries temporais
- 💾 **Alto custo computacional**: 480+ features por instância
- 📉 **Poucos samples**: Apenas 2,205 ciclos
- 🔧 **Feature engineering necessário**: Agregação/extração de features é essencial

**Ideal para:**
- Projetos de **detecção de anomalias** e **manutenção preditiva**
- Exploração de **deep learning** (LSTM, CNNs temporais, Transformers)
- Análise de **séries temporais multivariadas**
- **Publicação acadêmica** (dataset benchmark conhecido)

---

### ✅ **Dataset 2: Hydropower Turbines**
**Vantagens:**
- 📈 **Volume massivo**: 800K-1M+ instâncias (excelente para ML)
- 🌍 **Dados reais**: Coletados de turbinas em operação real (2022-2024)
- 🎯 **Target claro**: Classificação binária (presença de Glycerol)
- 💡 **Features agregadas**: Já processadas (avg, min, max) - prontas para uso
- ⚡ **Baixa complexidade**: 10 features numéricas apenas
- 🔄 **Séries temporais longas**: Permite análise temporal se necessário

**Desvantagens:**
- ⚠️ **Possível desbalanceamento**: Classes podem estar desbalanceadas
- 📉 **Features limitadas**: Apenas 10 features (menos informação)
- 🔀 **Turbinas diferentes**: T1 e T2 têm features distintas
- 📊 **Target simples**: Apenas classificação binária

**Ideal para:**
- Projetos de **classificação binária** em escala
- **Produção real**: Simular sistema de monitoramento em operação
- **Benchmark de algoritmos**: Grande volume permite validação robusta
- **Aprendizado com dados reais** e desbalanceados

---

### ✅ **Dataset 3: Industrial Equipment (Kaggle)**
*Necessita download via Kaggle - `dnkumars/industrial-equipment-monitoring-dataset`*

**Potencial:**
- 🌐 **Comunidade Kaggle**: Benchmark estabelecido com discussões ativas
- 📚 **Documentação externa**: Kernels e soluções disponíveis  
- 🏆 **Competição/Benchmark**: Métricas e baselines já definidos
- 📊 **Dados de equipamentos industriais**: Aplicável a cenários reais
- 🔄 **Comparação comunitária**: Validação com abordagens de outros desenvolvedores

**Características (após download):**
- 📦 **Volume**: A determinar após análise exploratória
- 🔢 **Features**: Depende da estrutura do dataset
- 🎯 **Target**: Identificação automática durante análise
- 💾 **Formato**: CSV via kagglehub

**Considerações:**
- ⚠️ **Requer download**: Execute célula de kagglehub primeiro
- 📥 **Dependência externa**: Necessita autenticação Kaggle
- 🔍 **Análise necessária**: Estrutura descoberta durante EDA

**Ideal para:**
- Comparação com benchmarks estabelecidos na comunidade
- Aprendizado com soluções compartilhadas (Kaggle kernels)
- Validação de abordagem contra outros competidores
- Projetos que buscam estado da arte documentado

**Execute a célula de download do dataset Kaggle para análise detalhada**

---

## 📋 Próximos Passos

1. **Se escolher Hydropower:**
   - ✓ Análise de balanceamento de classes
   - ✓ Split temporal (train/val/test)
   - ✓ Baseline com modelos simples (Logistic Regression, Random Forest)
   - ✓ Feature engineering adicional se necessário
   - ✓ Tratamento de desbalanceamento (SMOTE, class weights)

2. **Se escolher Hydraulic Systems:**
   - ✓ Feature extraction de séries temporais (estatísticas, FFT, wavelets)
   - ✓ Redução de dimensionalidade (PCA, autoencoders)
   - ✓ Exploração de modelos temporais (LSTM, GRU, Transformers)
   - ✓ Escolha de target principal ou multi-task learning

3. **Download Industrial Equipment (opcional):**
   - ✓ Executar célula de download
   - ✓ Análise exploratória completa
   - ✓ Comparação com os outros dois datasets